<a href="https://colab.research.google.com/github/viktoriagajdosova/Enhancing-Meta-Research-in-Psychology-by-Generative-AI/blob/main/pipelines/01_embedding-for-psychometrics/content_validity_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: Installation, Imports, and GPU Setup

# 1. Install necessary libraries
!pip install openai google-genai numpy scikit-learn sentence-transformers torch --quiet

# 2. Imports
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import torch
import random
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import userdata

# Imports for Models
from openai import OpenAI
from google import genai
from sentence_transformers import SentenceTransformer

#  Determinism
def set_all_seeds(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        print(f"GPU Determinism set for True and seed {seed_value}.")
    else:
        print(f"Seed {seed_value} set for CPU.")

# 3. Set device to GPU (cuda)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

set_all_seeds(42)

print("---")
print(f"Core libraries installed and imported.")
print(f"Using device: {device} (For local models)")
print("---")

In [ ]:
# CELL 2: API/Model Initialization and Universal Embedding Function

# Essential imports
from google.colab import userdata
from openai import OpenAI
from google import genai
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity

# ⚠️ SWITCH: SELECT ONE PLATFORM
# Choose 'OPENAI', 'GEMINI', 'E5_LOCAL', or 'DWULFF_LOCAL'

API_PLATFORM = 'E5_LOCAL'

# API_PLATFORM = 'GEMINI'
# API_PLATFORM = 'OPENAI'
# API_PLATFORM = 'E5_LOCAL'
# API_PLATFORM = 'DWULFF_LOCAL'

# --- CLIENTS ---
client = None
MODEL_NAME = None
OUTPUT_DIMENSIONALITY = None
API_KEY = None

# --- MODEL CONFIGURATION ---
if API_PLATFORM == 'OPENAI':
    # 1. OpenAI Configuration
    API_KEY = userdata.get('OPENAI_API_KEY')
    MODEL_NAME = 'text-embedding-3-large'
    OUTPUT_DIMENSIONALITY = 3072
    client = OpenAI(api_key=API_KEY)

elif API_PLATFORM == 'GEMINI':
    # 2. Gemini Configuration
    API_KEY = userdata.get('GOOGLE_API_KEY')
    MODEL_NAME = 'gemini-embedding-001'
    try:
        client = genai.Client(api_key=API_KEY)
    except Exception as e:
        print(f"FATAL ERROR: Gemini client initialization failed: {e}")
        client = None

elif API_PLATFORM == 'E5_LOCAL':
    # 3. E5 Local Configuration
    API_KEY = None
    MODEL_NAME = 'intfloat/e5-large-v2'
    OUTPUT_DIMENSIONALITY = 1024

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    try:
        client = SentenceTransformer(MODEL_NAME, device=device)
        client.eval()
        print(f"E5 model '{MODEL_NAME}' successfully loaded on device: {device}")
    except Exception as e:
        print(f"FATAL ERROR: E5 model initialization failed: {e}")
        client = None

elif API_PLATFORM == 'DWULFF_LOCAL':
    # 4. DWULFF Local Configuration (Specialized model)
    API_KEY = None
    MODEL_NAME = 'dwulff/mpnet-personality'
    OUTPUT_DIMENSIONALITY = 768

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    try:
        client = SentenceTransformer(MODEL_NAME, device=device)
        client.eval()
        print(f"DWULFF model '{MODEL_NAME}' successfully loaded on device: {device}")
    except Exception as e:
        print(f"FATAL ERROR: DWULFF model initialization failed: {e}")
        client = None

else:
    raise ValueError("Invalid API_PLATFORM value. Choose 'OPENAI', 'GEMINI', 'E5_LOCAL', or 'DWULFF_LOCAL'.")

# --- CHECK AND UNIVERSAL FUNCTION ---

if client is None:
     print("Error: API Client/Model is not initialized.")
elif API_PLATFORM not in ['E5_LOCAL', 'DWULFF_LOCAL'] and not API_KEY:
    print(f"ERROR: API key for {API_PLATFORM} was not found.")

print(f"--- ACTIVE CONFIGURATION ---")
print(f"Platform: {API_PLATFORM}")
print(f"Model: {MODEL_NAME}")
print(f"Dimension: {OUTPUT_DIMENSIONALITY if OUTPUT_DIMENSIONALITY else 'Default'}")
print("----------------------------")


def get_embeddings_for_texts(texts: list) -> np.ndarray:
    """Generates embeddings for a list of texts using the selected API/Model."""
    if client is None:
         print("Error: API Client/Model is not initialized.")
         return np.array([])

    if API_PLATFORM == 'OPENAI':
        # OpenAI SDK call
        response = client.embeddings.create(
            input=texts,
            model=MODEL_NAME,
            dimensions=OUTPUT_DIMENSIONALITY
        )
        embeddings_list = [item.embedding for item in response.data]
        return np.array(embeddings_list)

    elif API_PLATFORM == 'GEMINI':
        try:
            response = client.models.embed_content(
                model=MODEL_NAME,
                contents=texts,
            )
            embeddings_list = [result.values for result in response.embeddings]
            return np.array(embeddings_list)


        except Exception as e:
            print(f"🔴 Error calling Gemini API: {e}")
            return np.array([])

    elif API_PLATFORM == 'E5_LOCAL':
        # E5 LOGIC: Requires 'query: ' prefix (Správne odsadenie)
        try:
            prefixed_texts = [f"query: {t}" for t in texts]
            embeddings_matrix = client.encode(
                prefixed_texts,
                convert_to_numpy=True,
                show_progress_bar=False,
                device=client.device
            )
            return embeddings_matrix

        except Exception as e:
            print(f"🔴 Error calling E5/SentenceTransformer: {e}")
            return np.array([])

    elif API_PLATFORM == 'DWULFF_LOCAL':
        # DWULFF LOGIC: Standard SentenceTransformer (Správne odsadenie)
        try:
            embeddings_matrix = client.encode(
                texts,
                convert_to_numpy=True,
                show_progress_bar=False,
                device=client.device
            )
            return embeddings_matrix

        except Exception as e:
            print(f"🔴 Error calling DWULFF/SentenceTransformer: {e}")
            return np.array([])

    return np.array([])

In [ ]:
# CELL 3: Data Definition, Embedding Generation, and Similarity Calculation (UNIFIED LOGIC)

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

symptom_embeddings_matrix = None
item_embeddings_matrix = None

print("--- 1. INITIALIZATION CHECK ---")
if 'get_embeddings_for_texts' not in locals() and 'get_embeddings_for_texts' not in globals():
    print("FATAL ERROR: get_embeddings_for_texts is not defined.")
    print("Please ensure CELL 2 (API/Model Initialization) was executed successfully.")
    symptom_embeddings_matrix = None
    item_embeddings_matrix = None
    raise RuntimeError("Missing model client function. Cannot proceed with analysis.")

# -----------------------------------------------------------------
# STEP A: DATA DEFINITION
# -----------------------------------------------------------------
print("\n--- STEP A: DATA DEFINITION ---")

# Questionnaire definition
data =  {
    "Text_Item":
    [
     "I often lose sleep because of long gaming sessions.",
    "I never play games in order to feel better.",
    "I have significantly increased the amount of time I play games over last year.",
    "When I am not gaming I feel more irritable.",
    "I have lost interest in other hobbies because of my gaming.",
    "I would like to cut down my gaming time but it’s difficult to do.",
    "I usually think about my next gaming session when I am not playing.",
    "I play games to help me cope with any bad feelings I might have.",
    "I need to spend increasing amounts of time engaged in playing games.",
    "I feel sad if I am not able to play games.",
    "I have lied to my family members because the amount of gaming I do.",
    "I do not think I could stop gaming.",
    "I think gaming has become the most time consuming activity in my life.",
    "I play games to forget about whatever’s bothering me.",
    "I often think that a whole day is not enough to do everything I need to do in-game.",
    "I tend to get anxious if I can’t play games for any reason.",
    "I think my gaming has jeopardised the relationship with my partner.",
    "I often try to play games less but find I cannot.",
    "I know my main daily activity (i.e., occupation, education, homemaker, etc.) has not been negatively affected by my gaming.",
    "I believe my gaming is negatively impacting on important areas of my life.",

   ]
}

item_count = len(data["Text_Item"])
data['Item_ID'] = list(range(1, item_count + 1))
df = pd.DataFrame(data)
df['Text_Item'] = df['Text_Item'].str.strip()


# DSM-5 Criteria Definition
Symptoms_DSM5 = {
"1":"Preoccupation with Internet games.",
"2":"Withdrawal symptoms when Internet gaming is taken away.",
"3":"Tolerance—the need to spend increasing amounts of time engaged in Internet games.",
"4":"Unsuccessful attempts to control the participation in Internet games.",
"5":"Loss of interests in previous hobbies and entertainment.",
"6":"Continued excessive use of Internet games despite knowledge of psychosocial problems.",
"7":"Has deceived family members, therapists, or others regarding the amount of Internet gaming.",
"8":"Use of Internet games to escape or relieve a negative mood.",
"9":"Has jeopardized or lost a significant relationship, job, or educational or career opportunity."
}

symptom_texts = [text.strip() for text in Symptoms_DSM5.values()]
symptom_names = list(Symptoms_DSM5.keys())

print(f"Questionnaire Items: {len(df)}. DSM-5 Criteria: {len(symptom_names)}.")


# -----------------------------------------------------------------
# STEP B: EMBEDDING GENERATION
# -----------------------------------------------------------------
print("\n--- STEP B: EMBEDDING GENERATION ---")

# Preskočíme generovanie, ak bola chyba v kontrole inicializácie
if 'symptom_embeddings_matrix' not in locals():
    symptom_embeddings_matrix = None
    item_embeddings_matrix = None
else:
    try:
        # 1. GENERATE FOR CRITERIA
        print(f"Generating Vectors for criteria ({len(symptom_texts)} items)...")
        symptom_embeddings_matrix = get_embeddings_for_texts(symptom_texts)
        print(f"Criteria matrix shape: {symptom_embeddings_matrix.shape}")

        # 2. GENERATE FOR QUESTIONNAIRE ITEMS
        print(f"Generating Vectors for items ({len(df)} items)...")
        item_texts = df['Text_Item'].tolist()
        item_embeddings_matrix = get_embeddings_for_texts(item_texts)
        print(f"Item matrix shape: {item_embeddings_matrix.shape}")

    except Exception as e:
        print(f"FATAL ERROR in STEP B: {e}")
        symptom_embeddings_matrix = None
        item_embeddings_matrix = None


# -----------------------------------------------------------------
# STEP C: SIMILARITY CALCULATION AND ASSIGNMENT
# -----------------------------------------------------------------
print("\n--- STEP C: SIMILARITY CALCULATION AND ASSIGNMENT ---")

if symptom_embeddings_matrix is not None and item_embeddings_matrix is not None and symptom_embeddings_matrix.size > 0:
    # Dimension check
    if item_embeddings_matrix.shape[1] != symptom_embeddings_matrix.shape[1]:
         print(f"CRITICAL DIMENSION MISMATCH: {item_embeddings_matrix.shape[1]} != {symptom_embeddings_matrix.shape[1]}.")
         exit()

    print(f"Calculating cosine similarity ({len(df)} x {len(symptom_names)})...")

    similarity_matrix = cosine_similarity(item_embeddings_matrix, symptom_embeddings_matrix)

    similarity_df = pd.DataFrame(
        similarity_matrix,
        index=df.index,
        columns=symptom_names
    )

    # Assign best match
    df['Best_Match_DSM5'] = similarity_df.idxmax(axis=1)
    df['Max_Similarity'] = similarity_df.max(axis=1)

    print("\nSuccessfully assigned the most similar DSM-5 criteria to each item.")

    # --- Setting Pandas options to display all rows and columns ---
    print("Setting display limits to show all rows...")

    # Force display of all rows (sets limit to None)
    pd.set_option('display.max_rows', None)
    # Force display of all columns (sets limit to None)
    pd.set_option('display.max_columns', None)
    # --- End of settings ---

    # Display the full DataFrame using display() and hiding the index
    print("\nDetailed Item Assignment:")
    display(df[['Item_ID', 'Text_Item', 'Best_Match_DSM5', 'Max_Similarity']].style.hide(axis="index"))

    # -----------------------------------------------------------------
    # STEP D: AGGREGATE SUMMARY BY BEST MATCH DSM-5
    # -----------------------------------------------------------------
    print("\n--- STEP D: AGGREGATE SUMMARY BY BEST MATCH DSM-5 ---")

    # 1. Calculate the mean 'Max_Similarity' for each 'Best_Match_DSM5' group
    dsm5_summary = df.groupby('Best_Match_DSM5')['Max_Similarity'].mean().reset_index()

    # 2. Rename the result column to 'Average_Max_Similarity'
    dsm5_summary = dsm5_summary.rename(columns={'Max_Similarity': 'Average_Max_Similarity'})

    # 3. Create a DataFrame with all 10 criteria (1 to 10)
    full_dsm5_index = pd.DataFrame({'Best_Match_DSM5': symptom_names})

    # 4. Merge the summary with the complete index; unassigned criteria will have NaN similarity
    dsm5_summary = pd.merge(full_dsm5_index, dsm5_summary, on='Best_Match_DSM5', how='left')

    # 5. Replace missing (NaN) values with zero (0)
    dsm5_summary['Average_Max_Similarity'] = dsm5_summary['Average_Max_Similarity'].fillna(0)

    # 6. Sort by the numerical order of the DSM-5 symptoms (1, 2, 3...)
    # Convert Best_Match_DSM5 to a numeric type for correct sorting
    dsm5_summary['Sort_Key'] = pd.to_numeric(dsm5_summary['Best_Match_DSM5'])
    dsm5_summary = dsm5_summary.sort_values(by='Sort_Key', ascending=True).drop(columns=['Sort_Key'])

    # 7. Display the summary table using display() and hiding the index
    print("\nSummary Table: Average Maximum Similarity by DSM-5 Criterion (All Criteria)")
    display(dsm5_summary.style.hide(axis="index"))

else:
    print("Analysis was not run because embeddings were not successfully generated.")